# 🕵️ ForensicFace AI — Sketch to Realistic Face Reconstruction
### A tool to help victims describe and reconstruct suspect faces

This notebook uses **Stable Diffusion with ControlNet (Canny/Scribble)** to convert hand-drawn or digital forensic sketches into photorealistic faces.

**Pipeline:**
1. Upload a forensic sketch (hand-drawn or digital)
2. ControlNet extracts the structural edges from the sketch
3. Stable Diffusion generates a photorealistic face matching those features
4. Multiple variations are produced for review

---

## 📦 Step 1: Install Dependencies

In [1]:
# Install all required packages
!pip install -q diffusers transformers accelerate controlnet-aux \
    torch torchvision opencv-python-headless Pillow ipywidgets \
    gradio xformers safetensors

print("✅ All packages installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.1 MB/s eta 0:00:00
✅ All packages installed successfully.


## 🔧 Step 2: Import Libraries & Setup

In [6]:
import torch
import numpy as np
import cv2
import gradio as gr
from PIL import Image, ImageEnhance, ImageFilter
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
    StableDiffusionImg2ImgPipeline,
)
from controlnet_aux import CannyDetector, HEDdetector
import warnings
import os
import io
import base64
from pathlib import Path

warnings.filterwarnings("ignore")

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float16
    print("✅ Apple Silicon MPS detected")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("⚠️  No GPU found. Running on CPU (slower). Consider using Google Colab with GPU.")

print(f"\n🖥️  Device: {DEVICE} | Dtype: {DTYPE}")

# Output directory
OUTPUT_DIR = Path("forensic_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"📁 Output folder: {OUTPUT_DIR.resolve()}")

✅ GPU detected: Tesla T4

🖥️  Device: cuda | Dtype: torch.float16
📁 Output folder: /content/forensic_outputs


## 🤖 Step 3: Load AI Models
> This will download ~5GB on first run. Subsequent runs use the cache.

In [3]:
print("⏳ Loading ControlNet (scribble) model...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-scribble",
    torch_dtype=DTYPE,
)
print("✅ ControlNet loaded.")

print("\n⏳ Loading Stable Diffusion 1.5 pipeline...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,  # Disabled for forensic/law enforcement use
)

# Use efficient scheduler
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)

# Memory optimization
if DEVICE == "cuda":
    pipe.enable_xformers_memory_efficient_attention()
    print("✅ xFormers memory optimization enabled.")

print("\n🚀 All models loaded and ready!")

⏳ Loading ControlNet (scribble) model...


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

✅ ControlNet loaded.

⏳ Loading Stable Diffusion 1.5 pipeline...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results

✅ xFormers memory optimization enabled.

🚀 All models loaded and ready!


## 🎨 Step 4: Core Processing Functions

In [4]:
def preprocess_sketch(image: Image.Image, method: str = "auto") -> Image.Image:
    """
    Preprocess the input sketch for ControlNet.

    Args:
        image: Input PIL Image (sketch)
        method: 'auto' | 'canny' | 'scribble' | 'raw'
    Returns:
        Processed control image (edges on black background)
    """
    # Resize to 512x512 (optimal for SD 1.5)
    image = image.convert("RGB").resize((512, 512), Image.LANCZOS)
    img_array = np.array(image)

    if method == "raw":
        # Return as-is (for already clean sketches)
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        _, binary = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY_INV)
        control = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)
        return Image.fromarray(control)

    elif method == "canny":
        # Canny edge detection — good for precise line sketches
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blurred, threshold1=50, threshold2=150)
        control = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
        return Image.fromarray(control)

    elif method == "scribble" or method == "auto":
        # Scribble/HED — great for rough hand-drawn sketches
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        # Adaptive thresholding to handle varied sketch styles
        adaptive = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 11, 2
        )
        # Dilate slightly to thicken lines
        kernel = np.ones((2, 2), np.uint8)
        dilated = cv2.dilate(adaptive, kernel, iterations=1)
        control = cv2.cvtColor(dilated, cv2.COLOR_GRAY2RGB)
        return Image.fromarray(control)


def build_prompt(description: str, age_range: str, skin_tone: str, gender: str) -> tuple[str, str]:
    """
    Build a detailed prompt from victim-provided description.
    Returns (positive_prompt, negative_prompt)
    """
    base_positive = (
        f"highly detailed photorealistic portrait of a {age_range} year old {gender} "
        f"with {skin_tone} skin tone, "
        f"{description}, "
        "professional police forensic composite, realistic human face, "
        "sharp facial features, studio lighting, high resolution, "
        "8k uhd, dslr, photographic, hyperrealistic"
    )

    negative = (
        "cartoon, anime, drawing, sketch, painting, illustration, "
        "blurry, low quality, deformed, ugly, bad anatomy, "
        "disfigured, mutated, extra limbs, watermark, signature, "
        "text, logo, unrealistic, artistic, abstract"
    )

    return base_positive, negative


def generate_faces(
    sketch_image: Image.Image,
    description: str = "",
    age_range: str = "30-40",
    skin_tone: str = "medium",
    gender: str = "person",
    preprocessing: str = "auto",
    num_variations: int = 4,
    guidance_scale: float = 7.5,
    controlnet_strength: float = 0.85,
    num_steps: int = 30,
    seed: int = -1,
) -> list[Image.Image]:
    """
    Main generation function: sketch → realistic face(s)
    """
    # Preprocess sketch
    control_image = preprocess_sketch(sketch_image, method=preprocessing)

    # Build prompts
    positive_prompt, negative_prompt = build_prompt(description, age_range, skin_tone, gender)

    # Handle seed
    generator = None
    if seed != -1:
        generator = torch.Generator(device=DEVICE).manual_seed(seed)

    results = []
    for i in range(num_variations):
        print(f"  Generating variation {i+1}/{num_variations}...")
        output = pipe(
            prompt=positive_prompt,
            negative_prompt=negative_prompt,
            image=control_image,
            num_inference_steps=num_steps,
            guidance_scale=guidance_scale,
            controlnet_conditioning_scale=controlnet_strength,
            generator=generator,
        )
        results.append(output.images[0])

    return results, control_image


def save_results(results: list, prefix: str = "suspect") -> list[str]:
    """Save all generated images to disk."""
    saved = []
    for i, img in enumerate(results):
        path = OUTPUT_DIR / f"{prefix}_variation_{i+1}.png"
        img.save(path)
        saved.append(str(path))
    print(f"✅ Saved {len(saved)} images to {OUTPUT_DIR}/")
    return saved


print("✅ All processing functions defined.")

✅ All processing functions defined.


## 🖥️ Step 5: Launch the Gradio UI

In [8]:
# ─── Custom CSS for forensic/law-enforcement aesthetic ───
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Rajdhani:wght@400;600;700&display=swap');

:root {
    --bg-primary: #0a0c10;
    --bg-secondary: #111620;
    --bg-card: #141a24;
    --accent-blue: #1e90ff;
    --accent-amber: #f5a623;
    --accent-red: #e74c3c;
    --text-primary: #e8eaf0;
    --text-muted: #7a8499;
    --border: #1e2a3a;
}

body, .gradio-container {
    background: var(--bg-primary) !important;
    font-family: 'Rajdhani', sans-serif !important;
    color: var(--text-primary) !important;
}

.gradio-container { max-width: 1400px !important; }

h1, h2, h3 { font-family: 'Rajdhani', sans-serif !important; letter-spacing: 0.05em; }

.gr-button-primary {
    background: linear-gradient(135deg, #1e3a6e, #1e90ff) !important;
    border: 1px solid var(--accent-blue) !important;
    font-family: 'Share Tech Mono', monospace !important;
    letter-spacing: 0.1em !important;
    font-size: 1.1rem !important;
    text-transform: uppercase !important;
    transition: all 0.3s ease !important;
}

.gr-button-primary:hover {
    background: linear-gradient(135deg, #1e90ff, #00cfff) !important;
    box-shadow: 0 0 20px rgba(30, 144, 255, 0.5) !important;
}

.gr-form, .gr-panel, .gr-box {
    background: var(--bg-card) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
}

label { font-family: 'Share Tech Mono', monospace !important; font-size: 0.85rem !important; color: var(--accent-blue) !important; text-transform: uppercase !important; letter-spacing: 0.08em !important; }

textarea, input[type=text], input[type=number] {
    background: var(--bg-secondary) !important;
    border: 1px solid var(--border) !important;
    color: var(--text-primary) !important;
    font-family: 'Rajdhani', sans-serif !important;
}

.warning-banner {
    background: linear-gradient(135deg, #1a0a00, #2d1500) !important;
    border: 1px solid var(--accent-amber) !important;
    border-radius: 6px;
    padding: 12px 16px;
    color: var(--accent-amber) !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 0.8rem !important;
    letter-spacing: 0.05em;
}

.status-bar {
    background: var(--bg-secondary);
    border: 1px solid var(--border);
    border-radius: 4px;
    padding: 8px 12px;
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.8rem;
    color: var(--accent-blue);
}
"""

# ─── Gradio Interface ───
def run_pipeline(
    sketch,
    description,
    age_range,
    skin_tone,
    gender,
    preprocessing_method,
    num_variations,
    guidance_scale,
    controlnet_strength,
    num_steps,
    seed,
):
    if sketch is None:
        return None, None, "❌ Please upload a sketch first."

    sketch_pil = Image.fromarray(sketch) if isinstance(sketch, np.ndarray) else sketch

    status = "⏳ Processing sketch and generating realistic faces..."

    try:
        results, control_image = generate_faces(
            sketch_image=sketch_pil,
            description=description,
            age_range=age_range,
            skin_tone=skin_tone,
            gender=gender,
            preprocessing=preprocessing_method,
            num_variations=int(num_variations),
            guidance_scale=guidance_scale,
            controlnet_strength=controlnet_strength,
            num_steps=int(num_steps),
            seed=int(seed),
        )

        saved_paths = save_results(results)
        status = f"✅ Generated {len(results)} variations | Saved to: {OUTPUT_DIR.resolve()}"

        return (
            np.array(control_image),  # Preprocessed control image
            [np.array(r) for r in results],  # Generated faces gallery
            status
        )

    except Exception as e:
        return None, None, f"❌ Error: {str(e)}"


# ─── Build Gradio App ───
with gr.Blocks(css=CUSTOM_CSS, title="ForensicFace AI", theme=gr.themes.Base()) as app:

    # ── Header ──
    gr.HTML("""
    <div style="text-align:center; padding: 30px 0 10px; border-bottom: 1px solid #1e2a3a; margin-bottom: 24px;">
        <div style="font-family: 'Share Tech Mono', monospace; color: #7a8499; font-size:0.8rem; letter-spacing:0.3em; margin-bottom:8px;">FORENSIC INTELLIGENCE SYSTEM</div>
        <h1 style="font-family: 'Rajdhani', sans-serif; font-size: 2.8rem; font-weight: 700; color: #e8eaf0; margin: 0; letter-spacing: 0.08em;">
            🕵️ FORENSICFACE <span style="color: #1e90ff;">AI</span>
        </h1>
        <p style="color: #7a8499; font-family: 'Rajdhani', sans-serif; font-size: 1.1rem; margin-top: 8px; letter-spacing:0.05em;">
            Sketch-to-Realistic Face Reconstruction for Victim Testimonies
        </p>
    </div>
    """)

    # ── Warning Banner ──
    gr.HTML("""
    <div class="warning-banner">
        ⚠️  LAW ENFORCEMENT USE ONLY &nbsp;|&nbsp;
        All generated composites are AI-approximations for investigative assistance only.
        They do not constitute legal evidence without forensic expert verification.
        Handle all victim information with strict confidentiality protocols.
    </div>
    """)

    gr.HTML("<div style='margin: 16px 0'></div>")

    with gr.Row():
        # ── Left Panel: Inputs ──
        with gr.Column(scale=1):
            gr.HTML('<div style="font-family: Share Tech Mono, monospace; color: #1e90ff; font-size: 0.8rem; letter-spacing: 0.2em; margin-bottom: 12px;">── INPUT PANEL ──</div>')

            sketch_input = gr.Image(
                label="Upload Forensic Sketch",
                type="pil",
                height=300,
            )

            description_input = gr.Textbox(
                label="Victim's Verbal Description (optional)",
                placeholder="e.g., sharp jawline, narrow eyes, prominent nose, short dark hair, thin lips, beard...",
                lines=3,
            )

            with gr.Row():
                gender_input = gr.Dropdown(
                    label="Perceived Gender",
                    choices=["male", "female", "person"],
                    value="male",
                )
                age_input = gr.Dropdown(
                    label="Approximate Age Range",
                    choices=["18-25", "25-35", "35-45", "45-55", "55-65", "65+"],
                    value="25-35",
                )

            skin_input = gr.Dropdown(
                label="Skin Tone",
                choices=["light", "fair", "medium", "olive", "tan", "brown", "dark brown", "dark"],
                value="medium",
            )

            preprocessing_input = gr.Radio(
                label="Sketch Preprocessing Method",
                choices=["auto", "scribble", "canny", "raw"],
                value="auto",
                info="'auto' works best for most hand-drawn sketches",
            )

            # Advanced Settings
            with gr.Accordion("⚙️ Advanced Generation Settings", open=False):
                num_variations_input = gr.Slider(
                    label="Number of Variations",
                    minimum=1, maximum=8, step=1, value=4,
                )
                guidance_input = gr.Slider(
                    label="Prompt Guidance Scale",
                    minimum=1.0, maximum=15.0, step=0.5, value=7.5,
                    info="Higher = more adherent to description",
                )
                cn_strength_input = gr.Slider(
                    label="ControlNet Strength (Sketch Fidelity)",
                    minimum=0.3, maximum=1.0, step=0.05, value=0.85,
                    info="Higher = closer to sketch structure",
                )
                steps_input = gr.Slider(
                    label="Inference Steps",
                    minimum=10, maximum=50, step=5, value=30,
                    info="More steps = higher quality, slower",
                )
                seed_input = gr.Number(
                    label="Seed (-1 = random)",
                    value=-1,
                )

            generate_btn = gr.Button(
                "🔍 GENERATE COMPOSITE FACES",
                variant="primary",
                size="lg",
            )

        # ── Right Panel: Outputs ──
        with gr.Column(scale=2):
            gr.HTML('<div style="font-family: Share Tech Mono, monospace; color: #1e90ff; font-size: 0.8rem; letter-spacing: 0.2em; margin-bottom: 12px;">── OUTPUT PANEL ──</div>')

            with gr.Row():
                control_output = gr.Image(
                    label="Preprocessed Sketch (ControlNet Input)",
                    height=200,
                )

            gallery_output = gr.Gallery(
                label="Generated Composite Faces",
                columns=2,
                height=480,
                object_fit="cover",
                show_label=True,
            )

            status_output = gr.Textbox(
                label="Status",
                interactive=False,
                elem_classes=["status-bar"],
            )

    # ── Footer ──
    gr.HTML("""
    <div style="text-align:center; margin-top: 32px; padding-top: 20px; border-top: 1px solid #1e2a3a;">
        <span style="font-family: Share Tech Mono, monospace; color: #7a8499; font-size: 0.75rem; letter-spacing: 0.1em;">
            FORENSICFACE AI &nbsp;|&nbsp; Powered by Stable Diffusion + ControlNet &nbsp;|&nbsp;
            FOR OFFICIAL INVESTIGATIVE USE ONLY
        </span>
    </div>
    """)

    # ── Event Binding ──
    generate_btn.click(
        fn=run_pipeline,
        inputs=[
            sketch_input,
            description_input,
            age_input,
            skin_input,
            gender_input,
            preprocessing_input,
            num_variations_input,
            guidance_input,
            cn_strength_input,
            steps_input,
            seed_input,
        ],
        outputs=[
            control_output,
            gallery_output,
            status_output,
        ],
    )

print("🚀 Launching ForensicFace AI...")
app.launch(share=True, debug=False)


🚀 Launching ForensicFace AI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e219e6593f8f34bb2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🧪 Step 6: Programmatic Test (without UI)
Run this cell to quickly test the pipeline with a synthetic sketch.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

# Create a simple test sketch (or load your own PNG)
# Replace with: test_sketch = Image.open("your_sketch.png")

def create_test_sketch():
    """Creates a minimal face sketch for quick pipeline testing."""
    img = Image.new("RGB", (512, 512), color=(255, 255, 255))
    import PIL.ImageDraw as ImageDraw
    draw = ImageDraw.Draw(img)
    # Face oval
    draw.ellipse([130, 60, 380, 360], outline=(0, 0, 0), width=5)
    # Eyes
    draw.ellipse([170, 140, 220, 175], outline=(0, 0, 0), width=4)
    draw.ellipse([290, 140, 340, 175], outline=(0, 0, 0), width=4)
    # Nose
    draw.line([(255, 170), (235, 240), (275, 240)], fill=(0, 0, 0), width=4)
    # Mouth
    draw.arc([200, 265, 310, 320], start=0, end=180, fill=(0, 0, 0), width=4)
    # Eyebrows
    draw.line([(160, 125), (225, 118)], fill=(0, 0, 0), width=5)
    draw.line([(285, 118), (350, 125)], fill=(0, 0, 0), width=5)
    return img

print("Running quick test...")
test_sketch = create_test_sketch()
# test_sketch = Image.open("/content/images.jpg")
print("Generating 2 face variations from test sketch...")
results, control_img = generate_faces(
    sketch_image=test_sketch,
    description="square jaw, high cheekbones, stubble beard",
    age_range="30-40",
    skin_tone="medium",
    gender="male",
    preprocessing="scribble",
    num_variations=2,
    guidance_scale=7.5,
    controlnet_strength=0.8,
    num_steps=25,
)

# Display results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0a0c10')

titles = ["Input Sketch", "ControlNet Input", "Generated Face 1"]
images_show = [test_sketch, control_img, results[0]]

for ax, img, title in zip(axes, images_show, titles):
    ax.imshow(np.array(img))
    ax.set_title(title, color='#1e90ff', fontsize=13, fontweight='bold')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2a3a')

plt.suptitle("ForensicFace AI — Pipeline Test", color='#e8eaf0', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_result.png", dpi=150, bbox_inches='tight', facecolor='#0a0c10')
plt.show()

print(f"\n✅ Test complete! Output saved to {OUTPUT_DIR}/test_result.png")

---
## 📋 Model Architecture Overview

| Component | Model | Purpose |
|-----------|-------|----------|
| **Backbone** | Stable Diffusion v1.5 | Photorealistic image generation |
| **Control** | ControlNet Scribble | Enforce sketch structure/pose |
| **Scheduler** | UniPC Multistep | Fast, high-quality sampling |
| **Preprocessing** | OpenCV Adaptive Threshold | Sketch-to-edge conversion |
| **UI** | Gradio Blocks | Interactive web interface |

### 🔬 How It Works
1. **Sketch Preprocessing** — The input sketch is converted to clean edge lines using adaptive thresholding or Canny detection
2. **ControlNet** — The edge map is fed to ControlNet which conditions the diffusion model on the spatial structure of the face
3. **Diffusion** — Stable Diffusion generates a photorealistic face that respects the structural constraints from the sketch
4. **Prompt Guidance** — Verbal descriptions from the victim fine-tune features like age, skin tone, gender, and specific facial attributes

### ⚖️ Legal & Ethical Notice
- AI composites are **investigative aids only** — not legal evidence
- Outputs require **verification by a certified forensic artist**
- All victim data must be handled per applicable **data protection laws**
- Generated images should **never be published** without proper legal authorization

---
*ForensicFace AI | Built with Stable Diffusion + ControlNet | For Law Enforcement Use Only*